<a href="https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/problem_notebooks/Problem04_Heart_Disease_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>&nbsp;&nbsp;<a href="https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/kadimetla/AIML_Learning_Gang/main/statistics/problem_notebooks/Problem04_Heart_Disease_Prediction.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

**Compatible with Google Colab, Kaggle, JupyterLab, VS Code, and Binder.**

| Platform | How to open |
|----------|-------------|
| **Google Colab** | Click the Colab badge above |
| **Kaggle** | Click the Kaggle badge above, or *New Notebook → File → Import Notebook* |
| **JupyterLab** | `pip install jupyterlab && jupyter lab` in your terminal |
| **VS Code** | Open with the Jupyter extension (`.ipynb` support built-in) |
| **Binder** | Visit mybinder.org and point it to this repo |

> **Requirements:** `numpy` `pandas` `matplotlib` `seaborn` `scikit-learn` `scipy`  
> Pre-installed on Colab and Kaggle. The next cell installs them if missing.


In [ ]:
# Install required packages — safe on Colab, Kaggle, or local environments
%pip install numpy pandas matplotlib seaborn scikit-learn scipy --quiet


# ❤️ Problem 04: Heart Disease Prediction
## *Can data save lives? A cardiologist's machine learning challenge*

---

> **Story:** A cardiologist wants to use routine patient data to flag who is at high risk of heart disease *before* symptoms worsen.  
> Every prediction has stakes. A missed case can cost a life. Let's build something that helps.


## 🔧 Section 0: Setup


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report,
                              calibration_curve)

# ── Color palette: red=disease, blue=healthy ──
COLORS = {0: '#3A7DC9', 1: '#D64045'}  # blue=no disease, red=disease
PALETTE = [COLORS[0], COLORS[1]]
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print('Setup complete ✓')


## 🏥 Section 1: Meet the Problem

### Why Heart Disease?

- **#1 cause of death globally** — approximately 18 million deaths per year (WHO, 2023)
- Heart disease is often **silent** — many patients show no symptoms until a major cardiac event
- Routine clinical tests (blood pressure, ECG, cholesterol) can reveal warning signs **years earlier**
- Early detection can reduce mortality by up to **30%** through lifestyle changes and medication

### Our Goal
Given results from routine clinical tests, **predict which patients have heart disease**.

### What We Have
918 patient records based on the structure of the Cleveland Heart Disease Dataset — one of the most studied datasets in medical ML.

### ⚠️ The Stakes
| Error Type | Consequence |
|---|---|
| **False Negative** (miss a sick patient) | Patient goes untreated — potentially fatal |
| **False Positive** (alarm a healthy patient) | Unnecessary tests, cost, anxiety — manageable |

> **Key insight:** In medical diagnosis, **we accept more false alarms to avoid missing real cases.**  
> This shapes every modeling decision we make.


In [ ]:
# ── Generate synthetic but clinically realistic dataset ──
np.random.seed(42)
n = 918

# Age: 28-77 years
age = np.random.normal(54, 9, n).clip(28, 77).astype(int)

# Sex: 0=Female, 1=Male (more males in heart disease studies)
sex = np.random.choice([0, 1], n, p=[0.32, 0.68])

# Chest pain type: 0=Typical Angina, 1=Atypical Angina, 2=Non-anginal, 3=Asymptomatic
chest_pain = np.random.choice([0, 1, 2, 3], n, p=[0.10, 0.17, 0.28, 0.45])

# Resting blood pressure (mmHg)
resting_bp = np.random.normal(132, 18, n).clip(80, 200).astype(int)

# Serum cholesterol (mg/dl)
cholesterol = np.random.normal(198, 109, n).clip(85, 564).astype(int)

# Fasting blood sugar > 120 mg/dl: 0=No, 1=Yes
fasting_bs = np.random.choice([0, 1], n, p=[0.76, 0.24])

# Resting ECG: 0=Normal, 1=ST-T wave abnormality, 2=Left ventricular hypertrophy
resting_ecg = np.random.choice([0, 1, 2], n, p=[0.60, 0.19, 0.21])

# Max heart rate achieved (60-202 bpm)
max_hr = np.random.normal(136, 25, n).clip(60, 202).astype(int)

# Exercise-induced angina: 0=No, 1=Yes
exercise_angina = np.random.choice([0, 1], n, p=[0.67, 0.33])

# ST depression induced by exercise (0-6.2)
st_depression = np.random.exponential(0.9, n).clip(0, 6.2).round(1)

# Slope of peak exercise ST segment: 0=Upsloping, 1=Flat, 2=Downsloping
st_slope = np.random.choice([0, 1, 2], n, p=[0.21, 0.41, 0.38])

# ── Target: heart disease present (0=No, 1=Yes) ──
# Built with realistic clinical relationships
risk_score = (
    (age - 40) / 30 * 0.8 +
    sex * 0.6 +
    (chest_pain == 3) * 1.5 +
    (resting_bp - 120) / 40 * 0.4 +
    fasting_bs * 0.5 +
    exercise_angina * 1.0 +
    st_depression * 0.4 +
    (st_slope == 1) * 0.4 +
    (st_slope == 2) * (-0.3) +
    (max_hr - 150) / 30 * (-0.5) +
    np.random.normal(0, 0.8, n)
)
target = (risk_score > 0.5).astype(int)

df = pd.DataFrame({
    'age': age, 'sex': sex, 'chest_pain_type': chest_pain,
    'resting_bp': resting_bp, 'cholesterol': cholesterol,
    'fasting_bs': fasting_bs, 'resting_ecg': resting_ecg,
    'max_hr': max_hr, 'exercise_angina': exercise_angina,
    'st_depression': st_depression, 'st_slope': st_slope,
    'heart_disease': target
})

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]} patients | Columns: {df.shape[1]} features')


In [ ]:
# ── First look at the data ──
print('=== First 10 patients ===')
df.head(10)


In [ ]:
# ── Class balance ──
disease_count = df['heart_disease'].sum()
no_disease_count = (df['heart_disease'] == 0).sum()
disease_pct = 100 * disease_count / len(df)

print(f'Total patients : {len(df)}')
print(f'Heart disease  : {disease_count} patients ({disease_pct:.1f}%)')
print(f'No heart disease: {no_disease_count} patients ({100-disease_pct:.1f}%)')
print()
print('Class balance is reasonably even — good for initial modeling.')


### 📋 Data Dictionary

| Feature | Clinical Meaning | Values / Range |
|---|---|---|
| `age` | Patient age in years | 28 – 77 |
| `sex` | Biological sex | 0 = Female, 1 = Male |
| `chest_pain_type` | Type of chest pain reported | 0 = Typical Angina, 1 = Atypical Angina, 2 = Non-anginal Pain, 3 = Asymptomatic |
| `resting_bp` | Blood pressure at rest (mmHg) | 80 – 200; normal < 120, hypertension ≥ 140 |
| `cholesterol` | Serum cholesterol (mg/dl) | 85 – 564; high risk > 240 |
| `fasting_bs` | Fasting blood sugar > 120 mg/dl | 0 = No, 1 = Yes (diabetes proxy) |
| `resting_ecg` | Resting electrocardiogram result | 0 = Normal, 1 = ST-T wave abnormality, 2 = Left ventricular hypertrophy |
| `max_hr` | Maximum heart rate achieved during stress test (bpm) | 60 – 202; higher = better cardiac fitness |
| `exercise_angina` | Chest pain triggered by exercise | 0 = No, 1 = Yes |
| `st_depression` | ST segment depression during exercise (mm) | 0 – 6.2; >2mm suggests ischemia |
| `st_slope` | Slope of peak exercise ST segment | 0 = Upsloping (benign), 1 = Flat (concerning), 2 = Downsloping (benign) |
| `heart_disease` | **Target**: heart disease present | 0 = No, 1 = Yes |


## 🔍 Section 2: Data Health Check

Before any analysis, we verify the data is trustworthy.


In [ ]:
# ── Structure and types ──
print('=== DataFrame Info ===')
df.info()


In [ ]:
# ── Descriptive statistics ──
print('=== Descriptive Statistics ===')
print('For each numeric column:')
print('  count = number of non-null values')
print('  mean  = average value')
print('  std   = standard deviation (spread around mean)')
print('  min/max = range')
print('  25%/50%/75% = quartiles (50% = median)')
print()
df.describe().round(2)


In [ ]:
# ── Missing values check ──
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(missing_df.index, missing_df['Missing %'],
              color=['#2ecc71' if v == 0 else '#e74c3c' for v in missing_df['Missing %']])
ax.set_title('Missing Values by Feature (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xlabel('Feature')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('No missing values found — clean dataset (aside from cholesterol=0 we will address in Section 3).')


In [ ]:
# ── Class balance ──
vc = df['heart_disease'].value_counts()
pct_disease = 100 * vc[1] / len(df)
print(f'Patients WITH heart disease   : {vc[1]} ({pct_disease:.1f}%)')
print(f'Patients WITHOUT heart disease: {vc[0]} ({100-pct_disease:.1f}%)')
print()
print(f'{pct_disease:.1f}% of patients have heart disease.')


📊 **CONCEPT: Class Imbalance**

If 55% of patients have disease, a *naive model that always says 'Yes'* gets 55% accuracy — but it catches **0% of healthy patients** and provides zero clinical value.

We need metrics beyond raw accuracy:
- **Recall (Sensitivity):** Of all sick patients, how many did we catch?
- **Precision:** Of all patients we flagged, how many were actually sick?
- **F1-score:** Harmonic mean of Precision and Recall
- **AUC-ROC:** Overall discriminative ability across all thresholds


## 📊 Section 3: Column-by-Column Analysis

Before we can predict heart disease, we need to **understand our patients** one measurement at a time.


### 3.1 Age


In [ ]:
# ── Age statistics ──
print('=== Age Statistics ===')
print(f'Mean   : {df["age"].mean():.1f} years')
print(f'Median : {df["age"].median():.1f} years')
print(f'Std Dev: {df["age"].std():.1f} years')
print(f'Range  : {df["age"].min()} – {df["age"].max()} years')
print(f'Patients over 60: {(df["age"] > 60).sum()} ({100*(df["age"] > 60).mean():.1f}%)')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram with KDE
axes[0].hist(df['age'], bins=25, color='steelblue', alpha=0.7, edgecolor='white', density=True)
df['age'].plot.kde(ax=axes[0], color='navy', linewidth=2)
axes[0].axvline(df['age'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean={df["age"].mean():.1f}')
axes[0].axvline(df['age'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median={df["age"].median():.1f}')
axes[0].set_title('Age Distribution (Histogram + KDE)', fontweight='bold')
axes[0].set_xlabel('Age (years)'); axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot
axes[1].boxplot(df['age'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Age Box Plot', fontweight='bold')
axes[1].set_ylabel('Age (years)'); axes[1].set_xticks([])

# Empirical CDF
sorted_age = np.sort(df['age'])
cdf = np.arange(1, len(sorted_age)+1) / len(sorted_age)
axes[2].plot(sorted_age, cdf, color='steelblue', linewidth=2)
axes[2].axhline(0.5, color='orange', linestyle='--', alpha=0.7, label='50th percentile')
axes[2].axhline(0.75, color='red', linestyle='--', alpha=0.7, label='75th percentile')
axes[2].set_title('Empirical CDF of Age', fontweight='bold')
axes[2].set_xlabel('Age (years)'); axes[2].set_ylabel('Cumulative Proportion')
axes[2].legend()

plt.suptitle('Age Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Read CDF insight
pct_under_50 = 100 * (df['age'] < 50).mean()
print(f'CDF insight: {pct_under_50:.1f}% of patients are under age 50.')
print(f'Average age is {df["age"].mean():.0f}. Heart disease risk increases sharply after age 45.')


📊 **CONCEPT: The CDF (Cumulative Distribution Function)**

The CDF answers: *'What fraction of patients are at or below a given value?'*

- Read off any percentile instantly: the 75th percentile age is where the CDF = 0.75
- Unlike a histogram, the CDF never requires choosing a bin width
- In medicine, we often care about thresholds: 'What % of patients have dangerously high BP?'


### 3.2 Sex


In [ ]:
sex_counts = df['sex'].value_counts().sort_index()
sex_labels = {0: 'Female', 1: 'Male'}
print('=== Sex Distribution ===')
for k, v in sex_counts.items():
    print(f'{sex_labels[k]}: {v} ({100*v/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([sex_labels[k] for k in sex_counts.index], sex_counts.values,
              color=['#FF69B4', '#4169E1'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sex_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val}\n({100*val/len(df):.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.set_title('Sex Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()
print('Males make up 68% of this dataset.')
print('Men have higher heart disease risk, especially before menopause.')


### 3.3 Chest Pain Type


In [ ]:
cp_labels = {0: 'Typical\nAngina', 1: 'Atypical\nAngina', 2: 'Non-anginal\nPain', 3: 'Asymptomatic'}
cp_counts = df['chest_pain_type'].value_counts().sort_index()
print('=== Chest Pain Type ===')
for k, v in cp_counts.items():
    print(f'{cp_labels[k].replace(chr(10)," "):25s}: {v:4d} ({100*v/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
colors_cp = ['#27AE60', '#F39C12', '#2980B9', '#C0392B']
bars = ax.bar([cp_labels[k] for k in cp_counts.index], cp_counts.values,
              color=colors_cp, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, cp_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val} ({100*val/len(df):.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_title('Chest Pain Type Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()
print('CLINICAL INSIGHT: Asymptomatic is paradoxically the most common and dangerous type.')
print('These patients feel NO pain but may still have severe underlying disease.')


### 3.4 Resting Blood Pressure


In [ ]:
print('=== Resting Blood Pressure ===')
print(f'Mean   : {df["resting_bp"].mean():.1f} mmHg')
print(f'Median : {df["resting_bp"].median():.1f} mmHg')
print(f'Std Dev: {df["resting_bp"].std():.1f} mmHg')
print(f'Normal (<120)         : {(df["resting_bp"] < 120).sum()} ({100*(df["resting_bp"] < 120).mean():.1f}%)')
print(f'Elevated (120-130)    : {((df["resting_bp"] >= 120) & (df["resting_bp"] < 130)).sum()}')
print(f'Stage 1 HTN (130-140) : {((df["resting_bp"] >= 130) & (df["resting_bp"] < 140)).sum()}')
print(f'Stage 2 HTN (≥140)    : {(df["resting_bp"] >= 140).sum()} ({100*(df["resting_bp"] >= 140).mean():.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with thresholds
axes[0].hist(df['resting_bp'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
for thresh, label, col in [(120, 'Normal', 'green'), (130, 'Elevated', 'orange'), (140, 'Stage 2 HTN', 'red')]:
    axes[0].axvline(thresh, color=col, linestyle='--', linewidth=1.5, label=label)
axes[0].set_title('Resting BP Distribution', fontweight='bold')
axes[0].set_xlabel('Blood Pressure (mmHg)'); axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

# Box plot
axes[1].boxplot(df['resting_bp'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].axhline(140, color='red', linestyle='--', linewidth=1.5, label='HTN threshold')
axes[1].set_title('Resting BP Box Plot', fontweight='bold')
axes[1].set_ylabel('Blood Pressure (mmHg)'); axes[1].set_xticks([])
axes[1].legend()

plt.suptitle('Resting Blood Pressure Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


📊 **CONCEPT: Clinical Thresholds**

In medicine, features often have **meaningful thresholds** established by clinical research:
- BP < 120: Normal
- BP 120–129: Elevated
- BP 130–139: Stage 1 Hypertension
- BP ≥ 140: Stage 2 Hypertension

Pure statistical analysis (mean, std) misses these critical breakpoints.  
**Domain knowledge must always complement data analysis.**


### 3.5 Cholesterol


In [ ]:
print('=== Cholesterol (Raw) ===')
print(f'Mean   : {df["cholesterol"].mean():.1f} mg/dl')
print(f'Median : {df["cholesterol"].median():.1f} mg/dl')
print(f'Std Dev: {df["cholesterol"].std():.1f} mg/dl')
print(f'High cholesterol (>240): {(df["cholesterol"] > 240).sum()} ({100*(df["cholesterol"] > 240).mean():.1f}%)')
zero_chol = (df['cholesterol'] == 0).sum()
print(f'\nCholesterol = 0: {zero_chol} patients')
if zero_chol > 0:
    print('  → Physiologically impossible! These are likely MISSING values coded as 0.')
else:
    print('  → No zero values in this synthetic dataset (good data quality).')


In [ ]:
# ── Handle cholesterol = 0 ──
df_clean = df.copy()
zero_mask = df_clean['cholesterol'] == 0
if zero_mask.sum() > 0:
    chol_median = df_clean.loc[~zero_mask, 'cholesterol'].median()
    df_clean.loc[zero_mask, 'cholesterol'] = np.nan
    df_clean['cholesterol'] = df_clean['cholesterol'].fillna(chol_median)
    print(f'Replaced {zero_mask.sum()} zero-cholesterol values with median ({chol_median:.0f} mg/dl)')
else:
    df_clean = df.copy()
    print('No zero-cholesterol values to handle.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Histogram + KDE
axes[0].hist(df_clean['cholesterol'], bins=30, color='#E67E22', alpha=0.7, edgecolor='white', density=True)
df_clean['cholesterol'].plot.kde(ax=axes[0], color='#8E44AD', linewidth=2)
axes[0].axvline(240, color='red', linestyle='--', linewidth=1.5, label='High risk (>240)')
axes[0].set_title('Cholesterol Distribution', fontweight='bold')
axes[0].set_xlabel('Cholesterol (mg/dl)'); axes[0].set_ylabel('Density')
axes[0].legend()
# Box plot with outlier markers
axes[1].boxplot(df_clean['cholesterol'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='#E67E22', alpha=0.6),
               flierprops=dict(marker='o', markerfacecolor='red', markersize=4))
axes[1].set_title('Cholesterol Box Plot (outliers in red)', fontweight='bold')
axes[1].set_ylabel('Cholesterol (mg/dl)'); axes[1].set_xticks([])
plt.suptitle('Serum Cholesterol Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
df = df_clean  # use cleaned version going forward


### 3.6 Fasting Blood Sugar


In [ ]:
fbs_counts = df['fasting_bs'].value_counts().sort_index()
fbs_labels = {0: 'Normal (≤120)', 1: 'Elevated (>120)'}
print('=== Fasting Blood Sugar ===')
for k, v in fbs_counts.items():
    print(f'{fbs_labels[k]:20s}: {v} ({100*v/len(df):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Pie
axes[0].pie(fbs_counts.values, labels=[fbs_labels[k] for k in fbs_counts.index],
            autopct='%1.1f%%', colors=['#27AE60', '#E74C3C'], startangle=90)
axes[0].set_title('Fasting Blood Sugar (Pie)', fontweight='bold')
# Bar
bars = axes[1].bar([fbs_labels[k] for k in fbs_counts.index], fbs_counts.values,
                   color=['#27AE60', '#E74C3C'], edgecolor='white')
for bar, val in zip(bars, fbs_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val}', ha='center', fontweight='bold')
axes[1].set_title('Fasting Blood Sugar (Bar)', fontweight='bold')
axes[1].set_ylabel('Count')
plt.suptitle('Fasting Blood Sugar Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('24% have elevated fasting blood sugar — a diabetes marker and heart disease risk factor.')


### 3.7 Resting ECG


In [ ]:
ecg_labels = {0: 'Normal', 1: 'ST-T Abnormality', 2: 'LV Hypertrophy'}
ecg_counts = df['resting_ecg'].value_counts().sort_index()
print('=== Resting ECG ===')
for k, v in ecg_counts.items():
    print(f'{ecg_labels[k]:22s}: {v} ({100*v/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([ecg_labels[k] for k in ecg_counts.index], ecg_counts.values,
              color=['#27AE60', '#E67E22', '#E74C3C'], edgecolor='white')
for bar, val in zip(bars, ecg_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val} ({100*val/len(df):.1f}%)', ha='center', fontweight='bold', fontsize=9)
ax.set_title('Resting ECG Results', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()
abnormal_pct = 100 * (df['resting_ecg'] > 0).mean()
print(f'{abnormal_pct:.1f}% show ECG abnormalities — an important diagnostic signal.')


### 3.8 Maximum Heart Rate


In [ ]:
print('=== Maximum Heart Rate ===')
print(f'Mean   : {df["max_hr"].mean():.1f} bpm')
print(f'Median : {df["max_hr"].median():.1f} bpm')
print(f'Std Dev: {df["max_hr"].std():.1f} bpm')
print(f'Range  : {df["max_hr"].min()} – {df["max_hr"].max()} bpm')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['max_hr'], bins=25, color='#16A085', alpha=0.7, edgecolor='white', density=True)
df['max_hr'].plot.kde(ax=axes[0], color='#1A5276', linewidth=2)
axes[0].axvline(df['max_hr'].mean(), color='red', linestyle='--', label=f'Mean={df["max_hr"].mean():.0f}')
axes[0].set_title('Max Heart Rate Distribution', fontweight='bold')
axes[0].set_xlabel('Max HR (bpm)'); axes[0].set_ylabel('Density')
axes[0].legend()
axes[1].boxplot(df['max_hr'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='#16A085', alpha=0.6))
axes[1].set_title('Max Heart Rate Box Plot', fontweight='bold')
axes[1].set_ylabel('Max HR (bpm)'); axes[1].set_xticks([])
plt.suptitle('Maximum Heart Rate Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Average max HR is {df["max_hr"].mean():.0f} bpm.')
print('Higher max heart rate generally indicates better cardiac fitness.')
print('Note: max HR decreases with age — this motivates feature engineering later.')


### 3.9 Exercise-Induced Angina


In [ ]:
ea_counts = df['exercise_angina'].value_counts().sort_index()
ea_labels = {0: 'No Angina', 1: 'Exercise Angina'}
print('=== Exercise-Induced Angina ===')
for k, v in ea_counts.items():
    print(f'{ea_labels[k]:20s}: {v} ({100*v/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([ea_labels[k] for k in ea_counts.index], ea_counts.values,
              color=['#27AE60', '#E74C3C'], edgecolor='white')
for bar, val in zip(bars, ea_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val} ({100*val/len(df):.1f}%)', ha='center', fontweight='bold')
ax.set_title('Exercise-Induced Angina', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()
print('33% experience chest pain during exercise — a significant warning sign.')


### 3.10 ST Depression


In [ ]:
print('=== ST Depression ===')
print(f'Mean     : {df["st_depression"].mean():.2f} mm')
print(f'Median   : {df["st_depression"].median():.2f} mm')
skew = stats.skew(df['st_depression'])
print(f'Skewness : {skew:.2f} (positive = right-skewed)')
print(f'% with ST depression > 2mm: {100*(df["st_depression"]>2).mean():.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Raw histogram
axes[0].hist(df['st_depression'], bins=30, color='#8E44AD', alpha=0.7, edgecolor='white')
axes[0].set_title('ST Depression (Raw)', fontweight='bold')
axes[0].set_xlabel('ST Depression (mm)'); axes[0].set_ylabel('Count')
# Log-transformed
log_st = np.log1p(df['st_depression'])
axes[1].hist(log_st, bins=30, color='#2980B9', alpha=0.7, edgecolor='white')
axes[1].set_title('ST Depression (log(1+x) transformed)', fontweight='bold')
axes[1].set_xlabel('log(1 + ST Depression)'); axes[1].set_ylabel('Count')
# Box plot
axes[2].boxplot(df['st_depression'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='#8E44AD', alpha=0.6),
               flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
axes[2].set_title('ST Depression Box Plot (outliers)', fontweight='bold')
axes[2].set_ylabel('ST Depression (mm)'); axes[2].set_xticks([])
plt.suptitle('ST Depression Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Right-skewed: most patients have low ST depression, but a high-risk tail shows extreme values.')
print('Log transform makes the distribution more symmetric — useful for some models.')


### 3.11 ST Slope


In [ ]:
slope_labels = {0: 'Upsloping\n(Benign)', 1: 'Flat\n(Concerning)', 2: 'Downsloping\n(Benign)'}
slope_counts = df['st_slope'].value_counts().sort_index()
print('=== ST Slope ===')
for k, v in slope_counts.items():
    print(f'{slope_labels[k].replace(chr(10)," "):22s}: {v} ({100*v/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([slope_labels[k] for k in slope_counts.index], slope_counts.values,
              color=['#27AE60', '#E74C3C', '#F39C12'], edgecolor='white')
for bar, val in zip(bars, slope_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val} ({100*val/len(df):.1f}%)', ha='center', fontweight='bold', fontsize=9)
ax.set_title('ST Slope Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()
print('Upsloping ST is typically benign; flat/downsloping suggests ischemia.')


## 🎯 Section 4: The Target Variable

**Who has heart disease?** Understanding what we're predicting.


In [ ]:
vc = df['heart_disease'].value_counts().sort_index()
print('=== Heart Disease Target ===')
print(f'No Heart Disease: {vc[0]} patients ({100*vc[0]/len(df):.1f}%)')
print(f'Heart Disease   : {vc[1]} patients ({100*vc[1]/len(df):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut chart
wedges, texts, autotexts = axes[0].pie(
    vc.values, labels=['No Disease', 'Heart Disease'],
    autopct='%1.1f%%', colors=['#3A7DC9', '#D64045'],
    startangle=90, wedgeprops=dict(width=0.5)
)
axes[0].set_title('Heart Disease Distribution (Donut)', fontweight='bold')

# Age KDE by disease status
for hd, color, label in [(0, '#3A7DC9', 'No Disease'), (1, '#D64045', 'Heart Disease')]:
    subset = df[df['heart_disease'] == hd]['age']
    subset.plot.kde(ax=axes[1], color=color, linewidth=2.5, label=f'{label} (n={len(subset)})')
axes[1].set_title('Age Distribution by Disease Status', fontweight='bold')
axes[1].set_xlabel('Age (years)'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Target Variable Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

mean_age_disease = df[df['heart_disease']==1]['age'].mean()
mean_age_no_disease = df[df['heart_disease']==0]['age'].mean()
print(f'Mean age with disease: {mean_age_disease:.1f} years')
print(f'Mean age without disease: {mean_age_no_disease:.1f} years')
print('The age distribution shifts right for disease patients — older patients are more affected.')


📊 **CONCEPT: Sensitivity vs Specificity**

In medical classification, two key metrics:

| Metric | Definition | Formula |
|---|---|---|
| **Sensitivity (Recall)** | Of all sick patients, how many did we *catch*? | TP / (TP + FN) |
| **Specificity** | Of all healthy patients, how many did we *correctly clear*? | TN / (TN + FP) |

These **trade off** against each other. Lowering the decision threshold catches more sick patients  
(higher sensitivity) but also alarms more healthy patients (lower specificity).

> For heart disease screening: **we prioritize sensitivity** — missing a case is far worse than a false alarm.


## 📈 Section 5: Feature vs Target

**Which clinical measurements predict heart disease?**


### 5.1 Age vs Heart Disease


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
data_by_class = [df[df['heart_disease']==0]['age'], df[df['heart_disease']==1]['age']]
bp = axes[0].boxplot(data_by_class, patch_artist=True, labels=['No Disease', 'Heart Disease'])
for patch, color in zip(bp['boxes'], ['#3A7DC9', '#D64045']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[0].set_title('Age by Disease Status (Box Plot)', fontweight='bold')
axes[0].set_ylabel('Age (years)')

# Violin plot
parts = axes[1].violinplot(data_by_class, positions=[1, 2], showmedians=True)
for pc, color in zip(parts['bodies'], ['#3A7DC9', '#D64045']):
    pc.set_facecolor(color); pc.set_alpha(0.6)
axes[1].set_title('Age by Disease Status (Violin)', fontweight='bold')
axes[1].set_ylabel('Age (years)')
axes[1].set_xticks([1, 2]); axes[1].set_xticklabels(['No Disease', 'Heart Disease'])

plt.suptitle('Age vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

m0 = df[df['heart_disease']==0]['age'].mean()
m1 = df[df['heart_disease']==1]['age'].mean()
print(f'Mean age - No Disease  : {m0:.1f} years')
print(f'Mean age - Heart Disease: {m1:.1f} years')
print(f'Patients with heart disease are on average {m1-m0:.1f} years older.')


### 5.2 Sex vs Heart Disease


In [ ]:
sex_disease = df.groupby('sex')['heart_disease'].mean() * 100
sex_labels = {0: 'Female', 1: 'Male'}
print('=== Disease Rate by Sex ===')
for k in sex_disease.index:
    n_group = (df['sex']==k).sum()
    print(f'{sex_labels[k]:8s}: {sex_disease[k]:.1f}% disease rate (n={n_group})')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([sex_labels[k] for k in sex_disease.index], sex_disease.values,
              color=['#FF69B4', '#4169E1'], edgecolor='white')
for bar, val in zip(bars, sex_disease.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Heart Disease Rate by Sex', fontweight='bold', fontsize=13)
ax.set_ylabel('Disease Rate (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


### 5.3 Chest Pain Type vs Heart Disease


In [ ]:
cp_labels_map = {0: 'Typical\nAngina', 1: 'Atypical\nAngina', 2: 'Non-anginal', 3: 'Asymptomatic'}
cp_disease = df.groupby('chest_pain_type')['heart_disease'].mean() * 100
print('=== Disease Rate by Chest Pain Type ===')
for k in cp_disease.index:
    print(f'{cp_labels_map[k].replace(chr(10)," "):20s}: {cp_disease[k]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
bars = axes[0].bar([cp_labels_map[k] for k in cp_disease.index], cp_disease.values,
                   color=['#27AE60','#F39C12','#2980B9','#C0392B'], edgecolor='white')
for bar, val in zip(bars, cp_disease.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Disease Rate by Chest Pain Type', fontweight='bold')
axes[0].set_ylabel('Disease Rate (%)')
axes[0].set_ylim(0, 100)

# Heatmap: chest_pain x sex
pivot = df.groupby(['chest_pain_type', 'sex'])['heart_disease'].mean() * 100
pivot_df = pivot.unstack(level='sex').rename(columns={0: 'Female', 1: 'Male'})
pivot_df.index = [cp_labels_map[i].replace('\n',' ') for i in pivot_df.index]
sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='Reds', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': 'Disease Rate (%)'})
axes[1].set_title('Disease Rate: Chest Pain × Sex', fontweight='bold')
axes[1].set_xlabel('Sex'); axes[1].set_ylabel('Chest Pain Type')

plt.suptitle('Chest Pain Type vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

asym_rate = cp_disease.get(3, float('nan'))
print(f'Asymptomatic patients have the HIGHEST disease rate ({asym_rate:.1f}%).')
print('Counterintuitive but clinically well-known: no pain ≠ no disease.')


### 5.4 Resting BP vs Heart Disease


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data_by_class = [df[df['heart_disease']==0]['resting_bp'], df[df['heart_disease']==1]['resting_bp']]
# Box
bp = axes[0].boxplot(data_by_class, patch_artist=True, labels=['No Disease', 'Heart Disease'])
for patch, color in zip(bp['boxes'], ['#3A7DC9', '#D64045']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[0].axhline(140, color='red', linestyle='--', label='HTN threshold')
axes[0].set_title('Resting BP by Disease Status', fontweight='bold')
axes[0].set_ylabel('BP (mmHg)'); axes[0].legend()
# KDE overlay
for hd, color, label in [(0,'#3A7DC9','No Disease'),(1,'#D64045','Heart Disease')]:
    df[df['heart_disease']==hd]['resting_bp'].plot.kde(ax=axes[1], color=color,
                                                        linewidth=2, label=label)
axes[1].axvline(140, color='red', linestyle='--', alpha=0.7, label='HTN')
axes[1].set_title('Resting BP KDE by Disease Status', fontweight='bold')
axes[1].set_xlabel('BP (mmHg)'); axes[1].set_ylabel('Density')
axes[1].legend()
plt.suptitle('Resting Blood Pressure vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Higher BP is associated with disease, but the overlap is large — BP alone is not diagnostic.')


### 5.5 Cholesterol vs Heart Disease


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
data_by_class = [df[df['heart_disease']==0]['cholesterol'], df[df['heart_disease']==1]['cholesterol']]
bp = ax.boxplot(data_by_class, patch_artist=True, labels=['No Disease', 'Heart Disease'])
for patch, color in zip(bp['boxes'], ['#3A7DC9', '#D64045']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.axhline(240, color='orange', linestyle='--', label='High risk threshold')
ax.set_title('Cholesterol by Disease Status', fontweight='bold', fontsize=13)
ax.set_ylabel('Cholesterol (mg/dl)'); ax.legend()
plt.tight_layout()
plt.show()
m0 = df[df['heart_disease']==0]['cholesterol'].mean()
m1 = df[df['heart_disease']==1]['cholesterol'].mean()
print(f'Mean cholesterol - No Disease  : {m0:.1f} mg/dl')
print(f'Mean cholesterol - Heart Disease: {m1:.1f} mg/dl')
if m1 < m0:
    print('CLINICAL NOTE: Disease patients may show lower cholesterol — known as the "cholesterol paradox"')
    print('in advanced disease. This is a documented clinical phenomenon.')


### 5.6 Max Heart Rate vs Heart Disease


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Box plot
data_by_class = [df[df['heart_disease']==0]['max_hr'], df[df['heart_disease']==1]['max_hr']]
bp = axes[0].boxplot(data_by_class, patch_artist=True, labels=['No Disease', 'Heart Disease'])
for patch, color in zip(bp['boxes'], ['#3A7DC9', '#D64045']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[0].set_title('Max HR by Disease Status', fontweight='bold')
axes[0].set_ylabel('Max HR (bpm)')

# Scatter: age vs max_hr colored by disease
for hd, color, label in [(0,'#3A7DC9','No Disease'),(1,'#D64045','Heart Disease')]:
    subset = df[df['heart_disease']==hd]
    axes[1].scatter(subset['age'], subset['max_hr'], c=color, alpha=0.4,
                   s=20, label=label)
axes[1].set_title('Age vs Max HR (colored by disease)', fontweight='bold')
axes[1].set_xlabel('Age (years)'); axes[1].set_ylabel('Max HR (bpm)')
axes[1].legend()

plt.suptitle('Max Heart Rate vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

m0 = df[df['heart_disease']==0]['max_hr'].mean()
m1 = df[df['heart_disease']==1]['max_hr'].mean()
print(f'Mean max HR - No Disease  : {m0:.1f} bpm')
print(f'Mean max HR - Heart Disease: {m1:.1f} bpm')
print(f'Difference: {m0-m1:.1f} bpm')


📊 **CONCEPT: Direction of Correlation**

Max HR has a **negative correlation** with heart disease — lower max HR = higher risk.

In machine learning, *direction* matters as much as magnitude:
- A feature with correlation -0.4 is just as useful as one with +0.4
- Logistic regression and tree models handle both directions automatically
- When interpreting: don't just ask *'is this correlated?'* but *'in which direction?'*


### 5.7 ST Depression vs Heart Disease


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

data_by_class = [df[df['heart_disease']==0]['st_depression'], df[df['heart_disease']==1]['st_depression']]
# Box
bp = axes[0].boxplot(data_by_class, patch_artist=True, labels=['No Disease', 'Heart Disease'])
for patch, color in zip(bp['boxes'], ['#3A7DC9', '#D64045']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[0].set_title('ST Depression by Disease Status', fontweight='bold')
axes[0].set_ylabel('ST Depression (mm)')
# Violin
parts = axes[1].violinplot(data_by_class, positions=[1,2], showmedians=True)
for pc, color in zip(parts['bodies'], ['#3A7DC9', '#D64045']):
    pc.set_facecolor(color); pc.set_alpha(0.6)
axes[1].set_title('ST Depression (Violin)', fontweight='bold')
axes[1].set_ylabel('ST Depression (mm)')
axes[1].set_xticks([1,2]); axes[1].set_xticklabels(['No Disease', 'Heart Disease'])

plt.suptitle('ST Depression vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

m0 = df[df['heart_disease']==0]['st_depression'].mean()
m1 = df[df['heart_disease']==1]['st_depression'].mean()
print(f'Mean ST depression - No Disease  : {m0:.2f} mm')
print(f'Mean ST depression - Heart Disease: {m1:.2f} mm')
print('Very clear separation — ST depression is a key diagnostic feature.')


### 5.8 Exercise Angina vs Heart Disease


In [ ]:
ea_disease = df.groupby('exercise_angina')['heart_disease'].mean() * 100
ea_labels = {0: 'No Angina', 1: 'Exercise Angina'}
print('=== Disease Rate by Exercise Angina ===')
for k in ea_disease.index:
    n_group = (df['exercise_angina']==k).sum()
    print(f'{ea_labels[k]:20s}: {ea_disease[k]:.1f}% disease rate (n={n_group})')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([ea_labels[k] for k in ea_disease.index], ea_disease.values,
              color=['#27AE60', '#E74C3C'], edgecolor='white')
for bar, val in zip(bars, ea_disease.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Heart Disease Rate: Exercise Angina', fontweight='bold', fontsize=13)
ax.set_ylabel('Disease Rate (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()
print(f'Exercise angina patients: {ea_disease.get(1,0):.1f}% disease rate')
print(f'No exercise angina: {ea_disease.get(0,0):.1f}% disease rate')
print('Exercise angina is one of the strongest categorical predictors.')


### 5.9 ST Slope vs Heart Disease


In [ ]:
slope_labels_map = {0: 'Upsloping', 1: 'Flat', 2: 'Downsloping'}
slope_disease = df.groupby('st_slope')['heart_disease'].mean() * 100
print('=== Disease Rate by ST Slope ===')
for k in slope_disease.index:
    print(f'{slope_labels_map[k]:15s}: {slope_disease[k]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Disease rate bar
bars = axes[0].bar([slope_labels_map[k] for k in slope_disease.index], slope_disease.values,
                   color=['#27AE60','#E74C3C','#F39C12'], edgecolor='white')
for bar, val in zip(bars, slope_disease.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Disease Rate by ST Slope', fontweight='bold')
axes[0].set_ylabel('Disease Rate (%)')
axes[0].set_ylim(0, 100)

# Stacked bar
slope_cross = df.groupby(['st_slope','heart_disease']).size().unstack(fill_value=0)
slope_cross_pct = slope_cross.div(slope_cross.sum(axis=1), axis=0) * 100
slope_cross_pct.index = [slope_labels_map[i] for i in slope_cross_pct.index]
slope_cross_pct.plot(kind='bar', stacked=True, ax=axes[1],
                     color=['#3A7DC9','#D64045'], edgecolor='white', linewidth=0.5)
axes[1].set_title('Disease Proportion by ST Slope (Stacked)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].set_xlabel('ST Slope')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['No Disease','Heart Disease'])

plt.suptitle('ST Slope vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 🔗 Section 6: Multi-Variable Relationships

**The full picture** — how features relate to each other and to disease.


In [ ]:
# ── Correlation matrix ──
corr = df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            ax=ax, linewidths=0.5, mask=False,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Correlation with heart_disease (sorted) ===')
heart_corr = corr['heart_disease'].drop('heart_disease').sort_values(ascending=False)
for feat, val in heart_corr.items():
    direction = 'POSITIVE' if val > 0 else 'NEGATIVE'
    print(f'  {feat:20s}: {val:+.3f}  ({direction})')


**Reading the correlation matrix:**

- The `heart_disease` row/column shows each feature's correlation with our target
- **Positive** (red): higher value → more disease
- **Negative** (blue): higher value → less disease (e.g. max_hr — higher fitness = lower risk)
- Feature-to-feature correlations reveal redundancy (multicollinearity)


In [ ]:
# ── Top predictors by absolute correlation ──
abs_corr = heart_corr.abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#D64045' if heart_corr[feat] > 0 else '#3A7DC9' for feat in abs_corr.index]
bars = ax.barh(abs_corr.index, abs_corr.values, color=colors, edgecolor='white')
for bar, feat in zip(bars, abs_corr.index):
    val = heart_corr[feat]
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', fontsize=9)
red_patch = mpatches.Patch(color='#D64045', label='Positive correlation')
blue_patch = mpatches.Patch(color='#3A7DC9', label='Negative correlation')
ax.legend(handles=[red_patch, blue_patch])
ax.set_title('Feature Importance Preview: |Correlation| with Heart Disease',
             fontweight='bold', fontsize=12)
ax.set_xlabel('|Pearson Correlation|')
plt.tight_layout()
plt.show()
top_feat = abs_corr.index[-1]
print(f'Top predictor by correlation: {top_feat} (|r|={abs_corr.iloc[-1]:.3f})')


In [ ]:
# ── Pair plot: key features ──
key_features = ['age', 'max_hr', 'st_depression', 'resting_bp', 'cholesterol', 'heart_disease']
pair_df = df[key_features].copy()
pair_df['heart_disease'] = pair_df['heart_disease'].map({0: 'No Disease', 1: 'Heart Disease'})

g = sns.pairplot(pair_df, hue='heart_disease', palette={'No Disease':'#3A7DC9','Heart Disease':'#D64045'},
                 plot_kws={'alpha': 0.4, 's': 15}, diag_kind='kde', corner=True)
g.fig.suptitle('Pair Plot: Key Clinical Features', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Diagonal: KDE distribution per class.')
print('Off-diagonal: scatter of feature pairs — look for class separation.')


In [ ]:
# ── Risk Factor Interaction: Age × Max HR ──
fig, ax = plt.subplots(figsize=(9, 6))
for hd, color, label in [(0,'#3A7DC9','No Disease'),(1,'#D64045','Heart Disease')]:
    subset = df[df['heart_disease']==hd]
    ax.scatter(subset['age'], subset['max_hr'], c=color, alpha=0.5, s=25, label=label)
# Rough decision boundary
ax.plot([40, 77], [180, 100], 'k--', linewidth=1.5, alpha=0.5, label='Approx. boundary')
ax.set_title('Age vs Max HR — Disease Pattern', fontweight='bold', fontsize=13)
ax.set_xlabel('Age (years)'); ax.set_ylabel('Max Heart Rate (bpm)')
ax.legend()
ax.annotate('High Risk Zone\n(old + low HR)', xy=(68, 85), fontsize=9,
            color='#D64045', fontweight='bold')
ax.annotate('Lower Risk\n(young + high HR)', xy=(30, 175), fontsize=9,
            color='#3A7DC9', fontweight='bold')
plt.tight_layout()
plt.show()
print('Older patients with lower max HR cluster strongly in the disease group.')
print('This interaction motivates the hr_reserve feature we will engineer next.')


In [ ]:
# ── Clinical Risk Score ──
# Count how many risk factors each patient has
df['risk_factors'] = (
    (df['age'] > 55).astype(int) +
    (df['sex'] == 1).astype(int) +
    (df['exercise_angina'] == 1).astype(int) +
    (df['st_depression'] > 1.5).astype(int) +
    (df['fasting_bs'] == 1).astype(int) +
    (df['chest_pain_type'] == 3).astype(int)
)
df['risk_level'] = pd.cut(df['risk_factors'], bins=[-1, 2, 4, 6],
                           labels=['Low (0-2)', 'Medium (3-4)', 'High (5-6)'])

risk_disease = df.groupby('risk_level', observed=True)['heart_disease'].mean() * 100
print('=== Disease Rate by Risk Level ===')
for level, rate in risk_disease.items():
    n = (df['risk_level']==level).sum()
    print(f'{str(level):15s}: {rate:.1f}% disease rate (n={n})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
bars = axes[0].bar(risk_disease.index.astype(str), risk_disease.values,
                   color=['#27AE60','#F39C12','#E74C3C'], edgecolor='white')
for bar, val in zip(bars, risk_disease.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Disease Rate by Clinical Risk Level', fontweight='bold')
axes[0].set_ylabel('Disease Rate (%)')
axes[0].set_ylim(0, 100)

# Heatmap: risk_level x chest_pain_type
pivot2 = df.groupby(['risk_level','chest_pain_type'], observed=True)['heart_disease'].mean() * 100
pivot2_df = pivot2.unstack()
pivot2_df.columns = ['Typical','Atypical','Non-anginal','Asymptomatic']
sns.heatmap(pivot2_df, annot=True, fmt='.0f', cmap='Reds', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': 'Disease Rate (%)'})
axes[1].set_title('Disease Rate: Risk Level × Chest Pain Type', fontweight='bold')

plt.suptitle('Clinical Risk Score Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

high_rate = risk_disease.get('High (5-6)', 0)
print(f'High-risk patients have {high_rate:.1f}% disease rate — clinical risk scores work.')


## ⚙️ Section 7: Feature Engineering

**Transforming raw measurements into better predictors.**  
Good features = simpler models that generalize better.


In [ ]:
# ── 7.1 Age Groups ──
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 44, 54, 64, 100],
                          labels=['Young (<45)', 'Middle (45-54)', 'Senior (55-64)', 'Elderly (>64)'])

ag_disease = df.groupby('age_group', observed=True)['heart_disease'].mean() * 100
ag_counts = df.groupby('age_group', observed=True).size()
print('=== Disease Rate by Age Group ===')
for grp in ag_disease.index:
    print(f'{str(grp):20s}: {ag_disease[grp]:.1f}% (n={ag_counts[grp]})')

fig, ax = plt.subplots(figsize=(8, 4))
colors_ag = ['#27AE60','#F39C12','#E67E22','#E74C3C']
bars = ax.bar(ag_disease.index.astype(str), ag_disease.values, color=colors_ag, edgecolor='white')
for bar, val in zip(bars, ag_disease.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Heart Disease Rate by Age Group', fontweight='bold', fontsize=13)
ax.set_ylabel('Disease Rate (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()
print('Disease rate climbs sharply in senior and elderly groups.')


In [ ]:
# ── 7.2 BP Category ──
df['bp_category'] = pd.cut(df['resting_bp'],
                            bins=[0, 119, 129, 139, 300],
                            labels=['Normal (<120)', 'Elevated (120-129)',
                                    'Stage 1 HTN (130-139)', 'Stage 2 HTN (≥140)'])

bp_disease = df.groupby('bp_category', observed=True)['heart_disease'].mean() * 100
bp_counts = df.groupby('bp_category', observed=True).size()
print('=== Disease Rate by BP Category ===')
for grp in bp_disease.index:
    print(f'{str(grp):25s}: {bp_disease[grp]:.1f}% (n={bp_counts[grp]})')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(bp_disease.index.astype(str), bp_disease.values,
              color=['#27AE60','#F1C40F','#E67E22','#E74C3C'], edgecolor='white')
for bar, val in zip(bars, bp_disease.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
ax.set_title('Heart Disease Rate by BP Category', fontweight='bold', fontsize=13)
ax.set_ylabel('Disease Rate (%)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()
print('Clinical BP thresholds create cleaner categories than raw continuous values.')


In [ ]:
# ── 7.3 Heart Rate Reserve ──
# Theoretical max HR = 220 - age
# HR Reserve = theoretical max - achieved max (lower = less capacity used = concerning)
df['hr_reserve'] = (220 - df['age']) - df['max_hr']

print('=== HR Reserve ===')
print(f'Mean HR Reserve: {df["hr_reserve"].mean():.1f} bpm')
r_reserve = df['hr_reserve'].corr(df['heart_disease'])
r_maxhr   = df['max_hr'].corr(df['heart_disease'])
print(f'Correlation with disease - max_hr    : {r_maxhr:+.3f}')
print(f'Correlation with disease - hr_reserve: {r_reserve:+.3f}')
print(f'HR Reserve improves correlation by {abs(r_reserve)-abs(r_maxhr):.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
for hd, color, label in [(0,'#3A7DC9','No Disease'),(1,'#D64045','Heart Disease')]:
    subset = df[df['heart_disease']==hd]
    ax.scatter(subset['age'], subset['hr_reserve'], c=color, alpha=0.4, s=20, label=label)
ax.set_title('Age vs HR Reserve (colored by disease)', fontweight='bold', fontsize=13)
ax.set_xlabel('Age (years)'); ax.set_ylabel('HR Reserve (bpm)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.4 ST Risk Binary Feature ──
df['st_risk'] = (df['st_depression'] > 1.5).astype(int)

st_risk_disease = df.groupby('st_risk')['heart_disease'].mean() * 100
print('=== Disease Rate by ST Risk ===')
print(f'ST Depression ≤ 1.5 (low risk) : {st_risk_disease.get(0,0):.1f}%')
print(f'ST Depression > 1.5 (high risk): {st_risk_disease.get(1,0):.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['ST ≤ 1.5 (Low Risk)', 'ST > 1.5 (High Risk)']
bars = ax.bar(labels, [st_risk_disease.get(0,0), st_risk_disease.get(1,0)],
              color=['#27AE60','#E74C3C'], edgecolor='white')
for bar, val in zip(bars, [st_risk_disease.get(0,0), st_risk_disease.get(1,0)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Disease Rate by ST Depression Risk Category', fontweight='bold', fontsize=12)
ax.set_ylabel('Disease Rate (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.5 Composite Risk Score ──
df['composite_risk'] = (
    0.3 * (df['age'] - 40) / 30 +
    0.2 * df['sex'] +
    0.3 * df['exercise_angina'] +
    0.3 * df['st_depression'] / 3 +
    0.2 * (df['chest_pain_type'] == 3).astype(int) +
    0.15 * df['fasting_bs'] +
    -0.25 * (df['max_hr'] - 100) / 80
)

print('=== Composite Risk Score ===')
print(f'Mean risk score - No Disease  : {df[df["heart_disease"]==0]["composite_risk"].mean():.3f}')
print(f'Mean risk score - Heart Disease: {df[df["heart_disease"]==1]["composite_risk"].mean():.3f}')

from sklearn.metrics import roc_curve, roc_auc_score
fpr_r, tpr_r, _ = roc_curve(df['heart_disease'], df['composite_risk'])
auc_r = roc_auc_score(df['heart_disease'], df['composite_risk'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Distribution by disease
for hd, color, label in [(0,'#3A7DC9','No Disease'),(1,'#D64045','Heart Disease')]:
    df[df['heart_disease']==hd]['composite_risk'].plot.kde(ax=axes[0], color=color,
                                                            linewidth=2, label=label)
axes[0].set_title('Composite Risk Score by Disease Status', fontweight='bold')
axes[0].set_xlabel('Composite Risk Score'); axes[0].legend()
# ROC
axes[1].plot(fpr_r, tpr_r, color='#8E44AD', linewidth=2.5, label=f'AUC = {auc_r:.3f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='Random (AUC=0.5)')
axes[1].set_title('ROC Curve: Composite Risk Score', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
plt.suptitle('Composite Risk Score Preview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Composite score alone achieves AUC = {auc_r:.3f} — decent even before ML modeling!')


In [ ]:
# ── 7.6 Log Cholesterol ──
df['log_cholesterol'] = np.log1p(df['cholesterol'])

skew_raw = stats.skew(df['cholesterol'])
skew_log = stats.skew(df['log_cholesterol'])
print(f'Cholesterol skewness (raw)            : {skew_raw:.3f}')
print(f'Cholesterol skewness (log-transformed): {skew_log:.3f}')
print('Log transform reduces skewness — closer to normal distribution.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['cholesterol'], bins=30, color='#E67E22', alpha=0.7, edgecolor='white', density=True)
df['cholesterol'].plot.kde(ax=axes[0], color='#8E44AD', linewidth=2)
axes[0].set_title(f'Raw Cholesterol (skew={skew_raw:.2f})', fontweight='bold')
axes[0].set_xlabel('Cholesterol (mg/dl)')
axes[1].hist(df['log_cholesterol'], bins=30, color='#27AE60', alpha=0.7, edgecolor='white', density=True)
df['log_cholesterol'].plot.kde(ax=axes[1], color='#1A5276', linewidth=2)
axes[1].set_title(f'Log Cholesterol (skew={skew_log:.2f})', fontweight='bold')
axes[1].set_xlabel('log(1 + Cholesterol)')
plt.suptitle('Cholesterol: Before vs After Log Transform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 🤖 Section 8: Building the Prediction

**A model that could save lives.**  
We now bring together everything we learned to build and evaluate predictive models.


In [ ]:
# ── 8.1 Prepare Features ──
# Use original features + engineered ones
feature_cols = [
    'age', 'sex', 'chest_pain_type', 'resting_bp', 'cholesterol',
    'fasting_bs', 'resting_ecg', 'max_hr', 'exercise_angina',
    'st_depression', 'st_slope',
    'hr_reserve', 'st_risk', 'log_cholesterol'
]

X = df[feature_cols].copy()
y = df['heart_disease'].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeatures used: {list(X.columns)}')


In [ ]:
# ── 8.2 Train/Test Split (stratified) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} patients ({y_train.mean()*100:.1f}% disease)')
print(f'Test set     : {X_test.shape[0]} patients ({y_test.mean()*100:.1f}% disease)')
print()
print('Stratified split ensures both sets have the same class balance.')
print('This is critical for medical problems — we must not accidentally skew either set.')


In [ ]:
# ── 8.3 Scale Numeric Features ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # fit only on training data!
print('Features scaled using StandardScaler (mean=0, std=1).')
print('Scaler is FIT on training data only — prevents data leakage from test set.')


In [ ]:
# ── 8.4 Train 4 Models ──
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (d=4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    # Logistic Regression uses scaled features; trees don't need scaling
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'auc'      : roc_auc_score(y_test, y_prob),
    }
    print(f'{name:25s}: AUC={results[name]["auc"]:.3f}  Recall={results[name]["recall"]:.3f}')


In [ ]:
# ── 8.5 Comparison Table ──
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : f"{r['accuracy']*100:.1f}%",
        'Precision': f"{r['precision']*100:.1f}%",
        'Recall'   : f"{r['recall']*100:.1f}%",
        'F1-Score' : f"{r['f1']*100:.1f}%",
        'ROC-AUC'  : f"{r['auc']:.3f}",
    }
    for name, r in results.items()
}).T

print('=== Model Comparison ===')
print(metrics_df.to_string())


In [ ]:
# ── 8.6 ROC Curves ──
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#3498DB', '#27AE60', '#E74C3C', '#9B59B6']
for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={r["auc"]:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.5, linewidth=1, label='Random Classifier')
ax.set_title('ROC Curves: All Models', fontweight='bold', fontsize=13)
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


📊 **CONCEPT: ROC Curve and AUC**

The **ROC (Receiver Operating Characteristic) Curve** plots:
- **Y-axis:** True Positive Rate = Recall = Sensitivity (catching sick patients)
- **X-axis:** False Positive Rate = 1 - Specificity (incorrectly alarming healthy patients)

At each possible decision threshold, we get one point on the curve:
- Threshold → 0: predict everyone sick (TPR=1, FPR=1)
- Threshold → 1: predict no one sick (TPR=0, FPR=0)

**AUC (Area Under the Curve):**
- AUC = 1.0: perfect classifier
- AUC = 0.5: no better than random guessing
- AUC = 0.85+: clinically useful for screening

> AUC is **threshold-independent** — it measures overall discriminative ability across all operating points.


In [ ]:
# ── 8.7 Best Model: Deep Dive ──
best_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_name]
print(f'Best model: {best_name} (AUC={best["auc"]:.3f})')

# Confusion matrix
cm = confusion_matrix(y_test, best['y_pred'])
tn, fp, fn, tp = cm.ravel()
print(f'\nConfusion Matrix:')
print(f'  True Positives (TP) - Correctly identified sick : {tp}')
print(f'  True Negatives (TN) - Correctly cleared healthy: {tn}')
print(f'  False Positives (FP)- False alarms             : {fp}')
print(f'  False Negatives (FN)- MISSED sick patients     : {fn}  ← dangerous')
print()
print(f'Recall (sensitivity): {tp/(tp+fn)*100:.1f}% — catching {tp} of {tp+fn} sick patients')
print(f'Specificity         : {tn/(tn+fp)*100:.1f}% — correctly clearing {tn} of {tn+fp} healthy patients')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix heatmap
cm_labels = np.array([
    [f'TN\n{tn}\n(Correctly Cleared)', f'FP\n{fp}\n(False Alarm)'],
    [f'FN\n{fn}\n(MISSED — Dangerous)', f'TP\n{tp}\n(Correctly Flagged)']
])
cm_vals = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm_vals, annot=cm_labels, fmt='', cmap='Blues', ax=axes[0],
            linewidths=2, linecolor='white', cbar=False,
            xticklabels=['Predicted: No Disease', 'Predicted: Disease'],
            yticklabels=['Actual: No Disease', 'Actual: Disease'])
axes[0].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')
# Highlight the FN cell in red
axes[0].add_patch(plt.Rectangle((0, 1), 1, 1, fill=True, color='#E74C3C', alpha=0.25))

# Threshold analysis
thresholds = np.linspace(0.1, 0.9, 50)
recalls = []
precisions = []
for t in thresholds:
    y_pred_t = (best['y_prob'] >= t).astype(int)
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))

axes[1].plot(thresholds, recalls, color='#E74C3C', linewidth=2.5, label='Recall (Sensitivity)')
axes[1].plot(thresholds, precisions, color='#3A7DC9', linewidth=2.5, label='Precision')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Default threshold=0.5')
axes[1].axvline(0.3, color='orange', linestyle='--', alpha=0.7, label='Low threshold=0.3 (higher recall)')
axes[1].set_title('Recall vs Precision at Different Thresholds', fontweight='bold')
axes[1].set_xlabel('Decision Threshold'); axes[1].set_ylabel('Score')
axes[1].legend(); axes[1].set_ylim(0, 1.05)

plt.suptitle(f'Best Model Deep Dive: {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('For heart disease, lower thresholds catch more sick patients at the cost of more false alarms.')
print('We prefer high recall — no missed cases — even if it means more follow-up tests.')


In [ ]:
# ── 8.8 Feature Importance ──
# Use Random Forest or Gradient Boosting (tree-based importance)
tree_name = 'Random Forest' if 'Random Forest' in results else 'Gradient Boosting'
tree_model = results[tree_name]['model']
importances = pd.Series(tree_model.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax,
                 color=['#D64045' if v >= importances.median() else '#3A7DC9' for v in importances.values])
ax.set_title(f'Feature Importance: {tree_name}', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.axvline(importances.median(), color='gray', linestyle='--', alpha=0.5, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

top3 = importances.sort_values(ascending=False).head(3)
print('Top 3 features by importance:')
for feat, imp in top3.items():
    print(f'  {feat:20s}: {imp:.4f}')
print()
print('Does the model agree with our EDA? Compare with the correlation analysis in Section 6.')


In [ ]:
# ── 8.9 Calibration Curve (best model) ──
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, best['y_prob'], n_bins=10
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mean_predicted_value, fraction_of_positives,
        's-', color='#D64045', linewidth=2, markersize=8, label=f'{best_name}')
ax.plot([0,1],[0,1],'k--', alpha=0.7, label='Perfect calibration')
ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color='gray')
ax.set_title(f'Calibration Curve: {best_name}', fontweight='bold', fontsize=13)
ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives (Actual)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('A well-calibrated model: when it predicts 70% risk, ~70% of those patients actually have disease.')
print('Calibration matters for medical applications — doctors use probabilities to make treatment decisions.')


## 🩺 Section 9: The Medical Perspective

### What We Learned — Summary Table

| Concept | How We Used It | Clinical Finding |
|---|---|---|
| Distribution analysis | Histogram, box plot, KDE per feature | Max HR and ST depression have the clearest class separation |
| CDF | Empirical CDF for age | Instantly read off: 'X% of patients are under 50' |
| Clinical thresholds | BP categories, ST risk binary | Domain knowledge creates better features than raw values |
| Correlation heatmap | Features vs target + feature pairs | ST depression = strongest positive; max HR = strongest negative |
| Feature engineering | Age groups, hr_reserve, composite score | Engineered features improve both interpretability and model performance |
| Classification models | Logistic Regression, Decision Tree, RF, GBM | Random Forest / Gradient Boosting achieve AUC ~0.85+ |
| ROC / AUC | All 4 models on same plot | AUC measures overall discriminative ability across all thresholds |
| Recall vs Precision | Threshold analysis | Lowering threshold increases recall — prioritize for medical screening |
| Confusion matrix | TP / TN / FP / FN breakdown | FN (missed cases) are the most dangerous error type |
| Calibration | Predicted probability vs actual rate | Well-calibrated models give trustworthy probability estimates |

---

### The Bottom Line


In [ ]:
# ── Final clinical impact summary ──
best_recall = best['recall']
n_test = len(y_test)
n_actual_disease = y_test.sum()
n_caught = int(best_recall * n_actual_disease)

# Scale to 1000 patients
scale = 1000 / n_test
n_in_1000 = round(n_actual_disease * scale)
n_caught_1000 = round(n_caught * scale)

print('=' * 60)
print('  CLINICAL IMPACT SUMMARY')
print('=' * 60)
print(f'Best model: {best_name}')
print(f'  AUC-ROC : {best["auc"]:.3f}')
print(f'  Recall  : {best_recall*100:.1f}%')
print(f'  Precision: {best["precision"]*100:.1f}%')
print()
print(f'In a clinic of 1,000 patients:')
print(f'  ~{n_in_1000} patients would actually have heart disease')
print(f'  Our model would correctly flag ~{n_caught_1000} of them')
print(f'  Missed: ~{n_in_1000 - n_caught_1000} patients would require traditional clinical follow-up')
print()
print('IMPORTANT: The model ASSISTS, it does not REPLACE the doctor.')
print('High-risk flags trigger further diagnostic workup (ECG, stress test, angiography).')
print('Low-risk scores provide clinicians with data-driven reassurance — not dismissal.')


### Key Takeaways for a Future ML Engineer

1. **Domain knowledge is irreplaceable.** The cholesterol paradox, asymptomatic chest pain,  
   and clinical BP thresholds were all insights that came from medicine, not math.

2. **Choose metrics for the problem.** Accuracy is misleading in imbalanced medical datasets.  
   Recall (catching every sick patient) is the goal.

3. **Feature engineering beats model complexity.** A simple Logistic Regression with good features  
   often outperforms a complex model with raw features.

4. **Every model is uncertain.** Calibration tells you how much to trust a probability estimate.  
   In medicine, that trust is the difference between treatment and watchful waiting.

5. **False negatives have a face.** Every missed heart disease case is a patient who may not  
   receive timely treatment. Keep that human context in every modeling decision.

---

*Notebook complete. Next: extend with deep learning on ECG signals, or deploy as a clinical decision support tool.*
